# E0006 — Nemotron 3.5 Lightning / Kaggle Gate A

**Purpose:** environment + attached-model inspection only.

This notebook **does not load the 30B model**, does not access ARC answers, and does not submit anything to the competition leaderboard.

Required settings: GPU L4 x4, Internet OFF, target model attached.

Note: NVIDIA's nominal 24 GB is decimal GB. Do not compare it to a 24 GiB threshold.


In [ ]:
from __future__ import annotations
import importlib.metadata, json, os, platform, shutil, sys
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
OUT = Path('/kaggle/working/e0006_gate_a_inspect.json')
MODEL_HINTS = ('nemotron', 'lightning')

def package_version(name):
    try: return importlib.metadata.version(name)
    except Exception: return None

def dir_size_bytes(root):
    total = 0
    for p in root.rglob('*'):
        try:
            if p.is_file(): total += p.stat().st_size
        except Exception: pass
    return total

def discover_hf_model_roots(root):
    found = set()
    if not root.exists(): return []
    tokenizer_names = {'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model'}
    for config in root.rglob('config.json'):
        parent = config.parent
        low = str(parent).lower()
        if any(h in low for h in MODEL_HINTS) and any((parent / n).exists() for n in tokenizer_names):
            found.add(parent)
    return sorted(found)

def inspect_cuda():
    report = {'torch_available': False, 'cuda_available': False, 'device_count': 0, 'devices': []}
    try:
        import torch
    except Exception as exc:
        report['error'] = f'{type(exc).__name__}: {exc}'
        return report
    report['torch_available'] = True
    report['torch_version'] = getattr(torch, '__version__', None)
    report['cuda_runtime'] = getattr(torch.version, 'cuda', None)
    report['cuda_available'] = bool(torch.cuda.is_available())
    if not report['cuda_available']: return report
    report['device_count'] = torch.cuda.device_count()
    for i in range(report['device_count']):
        props = torch.cuda.get_device_properties(i)
        report['devices'].append({
            'index': i, 'name': props.name,
            'total_memory_gib': round(props.total_memory / 1024**3, 3),
            'total_memory_gb_decimal': round(props.total_memory / 1e9, 3),
            'capability': list(torch.cuda.get_device_capability(i)),
        })
    return report

os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')

models = []
for root in discover_hf_model_roots(INPUT_ROOT):
    try: config = json.loads((root / 'config.json').read_text(encoding='utf-8'))
    except Exception as exc: config = {'_read_error': f'{type(exc).__name__}: {exc}'}
    weights = sorted(root.glob('*.safetensors'))
    models.append({
        'path': str(root), 'size_gib': round(dir_size_bytes(root) / 1024**3, 3),
        'safetensor_files': len(weights), 'architecture': config.get('architectures'),
        'model_type': config.get('model_type'), 'torch_dtype': config.get('torch_dtype'),
        'has_chat_template': (root / 'chat_template.jinja').exists() or (root / 'tokenizer_config.json').exists(),
    })

cuda = inspect_cuda()
devices = cuda.get('devices') or []
packages = {n: package_version(n) for n in ('torch','transformers','accelerate','vllm','flashinfer-python','safetensors','triton')}
checks = {
    'four_cuda_devices': bool(cuda.get('cuda_available') and cuda.get('device_count') == 4),
    'all_devices_are_l4': bool(len(devices) == 4 and all('L4' in str(d.get('name','')) for d in devices)),
    'all_devices_sm89_or_newer': bool(devices and all(tuple(d.get('capability') or (0,0)) >= (8,9) for d in devices)),
    'all_devices_expected_l4_memory': bool(devices and all(float(d.get('total_memory_gib',0)) >= 21.5 for d in devices)),
    'nemotron_model_found': bool(models),
    'model_has_weight_shards': bool(models and all(int(m.get('safetensor_files',0)) > 0 for m in models)),
    'input_root_exists': INPUT_ROOT.exists(),
}
status = 'PASS_GATE_A' if all(checks.values()) else 'BLOCKED_GATE_A'
dependency_readiness = {
    'vllm_installed': packages.get('vllm') is not None,
    'flashinfer_installed': packages.get('flashinfer-python') is not None,
}
disk = shutil.disk_usage('/kaggle/working')
report = {
    'experiment':'E0006','gate':'A_INSPECT_ONLY','status':status,'checks':checks,
    'dependency_readiness':dependency_readiness,
    'dependency_status': 'READY_FOR_GATE_B_IMPORT_PROBE' if all(dependency_readiness.values()) else 'OFFLINE_RUNTIME_BUNDLE_REQUIRED',
    'python':sys.version,'platform':platform.platform(),
    'offline_env':{'HF_HUB_OFFLINE':os.environ.get('HF_HUB_OFFLINE'),'TRANSFORMERS_OFFLINE':os.environ.get('TRANSFORMERS_OFFLINE')},
    'packages':packages,'cuda':cuda,'attached_model_candidates':models,
    'working_disk':{'total_gib':round(disk.total/1024**3,3),'free_gib':round(disk.free/1024**3,3)},
    'next_gate_rule':'Gate B requires PASS_GATE_A plus a validated offline vLLM/FlashInfer runtime. Do not install from internet in the competition-valid run.',
}
OUT.write_text(json.dumps(report, indent=2, sort_keys=True)+'\n', encoding='utf-8')
print(json.dumps(report, indent=2, sort_keys=True))
print(f'\nWROTE: {OUT}')
